In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b")

In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explaination: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f"generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content

    return {"joke": response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f"write an explanation for the joke - {state['joke']}"
    response = llm.invoke(prompt).content

    return {"explanation": response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)



In [10]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the workplace was all about “slicing” up responsibilities and “topping” the competition! 🍕😄'}

In [11]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the workplace was all about “slicing” up responsibilities and “topping” the competition! 🍕😄'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f41-ed06-628d-8006-9491f5493708'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-11-20T15:07:36.532238+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f41-d926-6cb3-8005-5aa3e9e777be'}}, tasks=(), interrupts=())

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the workplace was all about “slicing” up responsibilities and “topping” the competition! 🍕😄'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f41-ed06-628d-8006-9491f5493708'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-11-20T15:07:36.532238+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f41-d926-6cb3-8005-5aa3e9e777be'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the workplace was all about “slicing” up responsibilities and “topping” the competition! 🍕😄'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f41-d926-6cb3-8005-5aa3e9e777be'}}, metadata={'source': 'loop', 'step': 5, 'parents': {}}, 

In [14]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({"topic": "pasta"}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti get a promotion?  \n\nBecause it always knew how to *pasta* the competition and never got tangled up in office drama! 🍝😄'}

In [15]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?  \n\nBecause it always knew how to *pasta* the competition and never got tangled up in office drama! 🍝😄'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f49-7b83-6c95-8002-29b2753b1942'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-20T15:10:59.378293+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f49-638f-62c3-8001-d68e582b5a04'}}, tasks=(), interrupts=())

In [16]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?  \n\nBecause it always knew how to *pasta* the competition and never got tangled up in office drama! 🍝😄'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f49-7b83-6c95-8002-29b2753b1942'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-20T15:10:59.378293+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f49-638f-62c3-8001-d68e582b5a04'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?  \n\nBecause it always knew how to *pasta* the competition and never got tangled up in office drama! 🍝😄'}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f49-638f-62c3-8001-d68e582b5a04'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_

Time Travel


In [20]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1e4f40-0d79-676e-8000-cb6dd7aa7f2a"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1e4f40-0d79-676e-8000-cb6dd7aa7f2a'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-11-20T15:06:46.247819+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f40-0d66-60de-bfff-426da79d0aa1'}}, tasks=(PregelTask(id='7f328177-bac9-28f3-6418-a4069b4a51a0', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to *dough* some work and finally get a slice of the action! 🍕😄'}),), interrupts=())

In [21]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1e4f40-0d79-676e-8000-cb6dd7aa7f2a"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the workplace had great *slices* of responsibility, a *cheesy* culture, and the chance to *rise* to the top—so it could finally stop being a *delivery* driver and start *topping* the charts! 🍕😄'}

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the workplace had great *slices* of responsibility, a *cheesy* culture, and the chance to *rise* to the top—so it could finally stop being a *delivery* driver and start *topping* the charts! 🍕😄'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f73-d206-607b-8003-f90155fa5744'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-11-20T15:29:55.878309+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e4f73-b68b-62d3-8002-2c57ce30f756'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the workplace had great *slices* of responsibility, a *cheesy* culture, and the chance to *rise* to the top—so it could finally stop being a *delivery* driver and start *topping* the charts! 🍕😄'}, ne

Updating State


In [23]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1e4f75-afe8-6e2e-8000-103d08b15d7e'}}

In [ ]:
list(workflow.get_state_history(config1))

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

In [ ]:
list(workflow.get_state_history(config1))